<a href="https://colab.research.google.com/github/TheHighBrid/Fidelis/blob/vosr2-colab/notebooks/Fidelis_VOSR2_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Fidelis 2.1: fidelity-controlled restoration

This notebook runs VOSR 2.0 internally at 4x on a free interactive Colab GPU, then produces a source-faithful 2x photograph. The finishing stage keeps the original composition and optical depth while admitting generated detail only where the source supports it. Use **Runtime > Run all**. The first run installs the pinned CUDA 13 runtime and restarts Colab automatically. After it reconnects, use **Runtime > Run all** once more. Then select one or more source images. The finished images download as one ZIP.

The first run downloads about 7 GB of model data into the temporary Colab session. It does not use your Google Drive.

In [ ]:
# @title 1. Install and verify the VOSR 2.0 GPU environment
import importlib.metadata, json, os, shutil, subprocess, sys
from pathlib import Path

# Inspect PyTorch in a child process so an upgrade never mixes two CUDA runtimes.
probe_code = "import json, torch; print(json.dumps({'torch': torch.__version__, 'cuda': torch.version.cuda}))"
runtime = json.loads(subprocess.check_output([sys.executable, '-c', probe_code], text=True).strip().splitlines()[-1])
cuda_parts = tuple(int(part) for part in (runtime.get('cuda') or '0').split('.')[:2])
try:
    hub_version = importlib.metadata.version('huggingface-hub')
except importlib.metadata.PackageNotFoundError:
    hub_version = 'missing'
try:
    transformers_version = importlib.metadata.version('transformers')
except importlib.metadata.PackageNotFoundError:
    transformers_version = 'missing'

needs_restart = False
if cuda_parts < (13, 0):
    driver = subprocess.check_output(['nvidia-smi', '--query-gpu=driver_version', '--format=csv,noheader'], text=True).strip().splitlines()[0]
    if int(driver.split('.')[0]) < 580:
        raise RuntimeError(f'The assigned Colab driver {driver} cannot run CUDA 13. Reconnect to a newer GPU runtime.')
    print(f'Installing official PyTorch 2.13.0 CUDA 13.0 over {runtime["torch"]} CUDA {runtime["cuda"]}...')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'torch==2.13.0', 'torchvision==0.28.0', '--index-url', 'https://download.pytorch.org/whl/cu130'], check=True)
    needs_restart = True
if hub_version != '1.31.0' or transformers_version != '5.17.0':
    print(f'Installing compatible Hugging Face packages over hub {hub_version}, transformers {transformers_version}...')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'huggingface_hub==1.31.0', 'transformers==5.17.0'], check=True)
    needs_restart = True
if needs_restart:
    print('Pinned packages installed. Colab is restarting. After it reconnects, choose Runtime > Run all once more.')
    os.kill(os.getpid(), 9)

import torch

if not torch.cuda.is_available():
    raise RuntimeError('No Colab GPU is active. Choose Runtime > Change runtime type > T4 GPU, then run all again.')

gpu = torch.cuda.get_device_properties(0)
gpu_gb = gpu.total_memory / 1024**3
if gpu_gb < 14:
    raise RuntimeError(f'{gpu.name} has only {gpu_gb:.1f} GB VRAM. Fidelis requires a T4-class GPU with at least 14 GB.')
print(f'GPU: {gpu.name}, {gpu_gb:.1f} GB')
print(f'PyTorch: {torch.__version__} | wheel CUDA: {torch.version.cuda} | capability: sm_{gpu.major}{gpu.minor}')

# Verify which optimized attention kernel this runtime actually selects.
import torch.nn.functional as F
try:
    from torch.profiler import ProfilerActivity, profile
    q = torch.randn(1, 24, 256, 64, device='cuda', dtype=torch.float16)
    with profile(activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA]) as probe:
        F.scaled_dot_product_attention(q, q, q)
    torch.cuda.synchronize()
    sdpa_events = sorted({event.key for event in probe.key_averages() if any(token in event.key.lower() for token in ('scaled_dot', 'flash', 'efficient_attention'))})
    print('SDPA kernel probe:', ', '.join(sdpa_events) or 'backend name unavailable')
    del q, probe
except Exception as error:
    print('SDPA kernel probe unavailable:', type(error).__name__)

os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['CUDA_MODULE_LOADING'] = 'LAZY'

ROOT = Path('/content/fidelis-v2')
COMFY = ROOT / 'ComfyUI'
NODE = COMFY / 'custom_nodes' / 'ComfyUI-VOSR2'
COMFY_COMMIT = 'd43a5fa20c8547ff42d13232f589a06536c42b97'
NODE_COMMIT = 'c9450b611e6b0e854212b81fecde2da5d088c1bc'
VOSR_REFERENCE_COMMIT = '516f292b99cf23c76fdc33351e86dc4f97711fe8'

ROOT.mkdir(parents=True, exist_ok=True)
if not COMFY.exists():
    subprocess.run(['git', 'clone', '--filter=blob:none', 'https://github.com/comfyanonymous/ComfyUI.git', str(COMFY)], check=True)
subprocess.run(['git', '-C', str(COMFY), 'fetch', '--depth=1', 'origin', COMFY_COMMIT], check=True)
subprocess.run(['git', '-C', str(COMFY), 'checkout', '--detach', COMFY_COMMIT], check=True)

NODE.parent.mkdir(parents=True, exist_ok=True)
if not NODE.exists():
    subprocess.run(['git', 'clone', '--filter=blob:none', 'https://github.com/ylchen333/ComfyUI-VOSR2.git', str(NODE)], check=True)
subprocess.run(['git', '-C', str(NODE), 'fetch', '--depth=1', 'origin', NODE_COMMIT], check=True)
subprocess.run(['git', '-C', str(NODE), 'checkout', '--detach', NODE_COMMIT], check=True)

# ComfyUI lists the whole PyTorch family as unpinned dependencies. Keep the
# verified CUDA/Hugging Face stack above instead of letting pip replace it.
requirements = []
protected = {'torch', 'torchvision', 'torchaudio', 'transformers'}
for raw_line in (COMFY / 'requirements.txt').read_text().splitlines():
    package = raw_line.strip().split('=', 1)[0].split('>', 1)[0].split('<', 1)[0].split('[', 1)[0].lower()
    if package not in protected:
        requirements.append(raw_line)
filtered_requirements = ROOT / 'comfy-requirements-fidelis.txt'
filtered_requirements.write_text('\n'.join(requirements) + '\n')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(filtered_requirements)], check=True)

if tuple(int(part) for part in torch.version.cuda.split('.')[:2]) < (13, 0):
    raise RuntimeError(f'Expected CUDA 13 PyTorch, found {torch.__version__} / CUDA {torch.version.cuda}. Restart the runtime and run all again.')
from huggingface_hub import is_offline_mode
assert callable(is_offline_mode)
print('Environment ready. VOSR reference:', VOSR_REFERENCE_COMMIT)

GPU: Tesla T4, 14.6 GB
PyTorch: 2.13.0+cu130 | wheel CUDA: 13.0 | capability: sm_75
SDPA kernel probe: aten::_efficient_attention_forward, aten::_scaled_dot_product_efficient_attention, aten::scaled_dot_product_attention


In [ ]:
# @title 2. Select the original images
from google.colab import files

INPUT_DIR = ROOT / 'input'
OUTPUT_DIR = ROOT / 'output'
shutil.rmtree(INPUT_DIR, ignore_errors=True)
shutil.rmtree(OUTPUT_DIR, ignore_errors=True)
INPUT_DIR.mkdir(parents=True)
OUTPUT_DIR.mkdir(parents=True)

uploaded = files.upload()
allowed = {'.jpg', '.jpeg', '.png', '.webp', '.bmp', '.tif', '.tiff'}
saved = []
for original_name, data in uploaded.items():
    name = Path(original_name).name
    if Path(name).suffix.lower() not in allowed:
        print('Skipped unsupported file:', name)
        continue
    target = INPUT_DIR / name
    target.write_bytes(data)
    saved.append(target)
if not saved:
    raise RuntimeError('No supported image was selected.')
print(f'Ready to restore {len(saved)} image(s).')

In [ ]:
# @title 3. Restore internally at 4x, finish at natural 2x, and download
import gc, importlib, types, zipfile
import numpy as np
from PIL import Image, ImageFilter

os.chdir(COMFY)
sys.argv = [sys.argv[0], '--lowvram']
sys.path.insert(0, str(COMFY))
package_name = 'fidelis_vosr2'
package = types.ModuleType(package_name)
package.__path__ = [str(NODE)]
sys.modules[package_name] = package
loader = importlib.import_module(f'{package_name}.loader')
inference = importlib.import_module(f'{package_name}.inference')

print('Downloading and loading VOSR 2.0. The first run downloads about 7 GB...')
model = loader.load_vosr2('VOSR2', 'fp16')

def _blur(array, radius):
    encoded = Image.fromarray(np.clip(array * 255.0 + 0.5, 0, 255).astype(np.uint8))
    return np.asarray(encoded.filter(ImageFilter.GaussianBlur(radius)), dtype=np.float32) / 255.0

def fidelity_finish(original, generated):
    # Work at the useful delivery size. A 4x generative pass supplies candidate
    # detail, while the 2x source resize remains the structural reference.
    target = (original.width * 2, original.height * 2)
    base = np.asarray(original.resize(target, Image.Resampling.LANCZOS), dtype=np.float32) / 255.0
    candidate = np.asarray(generated.resize(target, Image.Resampling.LANCZOS), dtype=np.float32) / 255.0

    base_soft = _blur(base, 1.5)
    candidate_soft = _blur(candidate, 1.5)
    base_low = _blur(base, 8.0)
    candidate_low = _blur(candidate, 8.0)
    base_mid, candidate_mid = base_soft - base_low, candidate_soft - candidate_low
    base_high, candidate_high = base - base_soft, candidate - candidate_soft

    # Confidence falls when VOSR changes broad structure or creates an edge
    # unsupported by the source. This protects identity, garment construction,
    # distant faces, and the original depth of field.
    weights = np.array([0.2126, 0.7152, 0.0722], dtype=np.float32)
    base_edge = np.abs(base_mid @ weights) + np.abs(base_high @ weights)
    candidate_edge = np.abs(candidate_mid @ weights) + np.abs(candidate_high @ weights)
    edge_support = np.clip((base_edge + 0.0025) / (candidate_edge + 0.0025), 0.0, 1.0)
    low_disagreement = np.mean(np.abs(candidate_low - base_low), axis=2)
    agreement = np.exp(-np.square(low_disagreement / 0.085))
    confidence = _blur(np.repeat((edge_support * agreement)[..., None], 3, axis=2), 2.0)[..., :1]

    low_alpha = (0.03 + 0.07 * agreement)[..., None]
    mid_alpha = 0.06 + 0.34 * confidence
    high_alpha = 0.10 + 0.50 * confidence
    finished = (base + low_alpha * (candidate_low - base_low)
                + mid_alpha * (candidate_mid - base_mid)
                + high_alpha * (candidate_high - base_high))
    return Image.fromarray(np.clip(finished * 255.0 + 0.5, 0, 255).astype(np.uint8))

completed = []
for index, source in enumerate(saved, 1):
    print(f'[{index}/{len(saved)}] Restoring {source.name}')
    with Image.open(source) as opened:
        rgb = opened.convert('RGB')
        image = torch.from_numpy(np.asarray(rgb).copy()).float().div(255).unsqueeze(0)
    with torch.inference_mode():
        result = inference.run_vosr2(
            model=model, images_bhwc01=image, upscale=4, seed=42,
            color_alignment='wavelet', tile_size=512, tile_overlap=64,
            vae_tile_size=1024, vae_tile_overlap=128,
        )
    pixels = result[0].clamp(0, 1).mul(255).round().byte().cpu().numpy()
    generated = Image.fromarray(pixels)
    finished = fidelity_finish(rgb, generated)
    destination = OUTPUT_DIR / f'{source.stem}-fidelis-natural-2x.jpg'
    finished.save(destination, format='JPEG', quality=96, subsampling=0, optimize=True)
    completed.append(destination)
    del image, result, pixels, generated, finished
    gc.collect()
    torch.cuda.empty_cache()

archive = ROOT / 'fidelis-results.zip'
with zipfile.ZipFile(archive, 'w', compression=zipfile.ZIP_DEFLATED) as bundle:
    for output in completed:
        bundle.write(output, arcname=output.name)
print(f'Finished {len(completed)} image(s). Downloading {archive.name}...')
files.download(str(archive))